In [29]:
import rasterio as rio
import rioxarray as rxr
import odc.geo.xr
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm
from rasterio.enums import Resampling
import math

import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from joblib import Parallel, delayed
from dask.distributed import Client


def compute_transitions(tile_path: Path):
    raster_20 = rxr.open_rasterio(tile_path).squeeze()
    raster_21 = rxr.open_rasterio(tile_path.parent.parent / '2021' / tile_path.name).squeeze()
    raster_22 = rxr.open_rasterio(tile_path.parent.parent / '2022' / tile_path.name).squeeze()
    raster_23 = rxr.open_rasterio(tile_path.parent.parent / '2023' / tile_path.name).squeeze()
    raster_24 = rxr.open_rasterio(tile_path.parent.parent / '2024' / tile_path.name).squeeze()
    
    # if rasters have different shapes, reproject_match to the bounds of raster_20
    if raster_21.shape != raster_20.shape:
        raster_21 = raster_21.rio.reproject_match(raster_20)
    if raster_22.shape != raster_20.shape:
        raster_22 = raster_22.rio.reproject_match(raster_20)
    if raster_23.shape != raster_20.shape:
        raster_23 = raster_23.rio.reproject_match(raster_20)
    if raster_24.shape != raster_20.shape:
        raster_24 = raster_24.rio.reproject_match(raster_20)
    
    raster_stack = np.stack([raster_20, raster_21, raster_22, raster_23, raster_24], axis=0)
    transitions = np.zeros(raster_20.shape, dtype=np.uint8)
    for i in range(1, raster_stack.shape[0]):
        transitions += (raster_stack[i] != raster_stack[i - 1]).astype(np.uint8)
    transitions = raster_20.copy(data=transitions)
    transitions = transitions.rio.write_nodata(15).astype(np.uint8)
    
    out_path = tile_path.parent.parent / 'transitions' / tile_path.name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    transitions.rio.to_raster(out_path, compress='LZW', tiled=True)


# tile_paths = list(Path('../runs/s2_out/2020/').rglob('*.tif'))
# Parallel(n_jobs=4)(delayed(compute_transitions)(tile_path) for tile_path in tqdm(tile_paths))

In [10]:
footprint_gdf = gpd.read_file(r'd:\chesapeake_bay_2022_2024edition\footprint.gpkg').simplify(10).buffer(10)

geom_utm17 = footprint_gdf.to_crs('EPSG:32617').geometry.union_all()
geom_utm18 = footprint_gdf.to_crs('EPSG:32618').geometry.union_all()


In [ ]:
buffer = 16 # pixels to clip from each edge to remove edge artifacts

cmap = None
footprint_gdf = gpd.read_file(r'd:\chesapeake_bay_2022_2024edition\footprint.gpkg')

# for folder in ['2020', '2021', '2022', '2023', '2024', '2025']:
#     temp_out_path = Path('D:/s2_out/') / folder / 'clipped'
#     temp_out_path.mkdir(parents=True, exist_ok=True)
#     for file in tqdm(list(Path('D:/s2_out/').glob(f'{folder}/*.tif')), desc=f'Processing {folder}'):
#         if cmap is None:
#             with rio.open(file) as src:
#                 cmap = src.colormap(1)
#         # clip around the edges to remove edge artifacts and save to temp folder
#         raster = rxr.open_rasterio(file).squeeze()
#         clipped = raster.isel(x=slice(buffer, -buffer), y=slice(buffer, -buffer))
#         # if the raster name starts with 17, it's in UTM 17, otherwise it's in UTM 18
#         utm_zone = int(file.name[:2])
#         if utm_zone == 17:
#             clipped = clipped.odc.crop(geom_utm17)
#         elif utm_zone == 18:
#             clipped = clipped.odc.crop(geom_utm18)
#         # clipped = clipped.rio.clip(footprint_gdf.to_crs(footprint_gdf).geometry, drop=True)
#         clipped.rio.to_raster(temp_out_path / file.name, compress='LZW', tiled=True)
#         with rio.open(temp_out_path / file.name, 'r+') as src:
#             src.write_colormap(1, cmap)
#             src.build_overviews([2, 4, 8, 16], Resampling.nearest)

def get_overview_levels(src, min_tile_size=16):
    min_dim = min(src.width, src.height)
    max_pow = int(math.log2(min_dim / min_tile_size))
    levels = [2**i for i in range(1, max_pow + 1)] if max_pow > 0 else []
    print(f'Overview levels for {src.name}: {levels}')
    return levels

for folder in ['transitions']:
    temp_out_path = Path('D:/s2_out/') / folder / 'clipped'
    temp_out_path.mkdir(parents=True, exist_ok=True)

    def process_file(file, temp_out_path=temp_out_path, cmap=cmap):
        
        # grab colormap from the first file
        with rio.open(file) as src:
            try:
                cmap = src.colormap(1)
            except ValueError:
                cmap = None
            
        # clip around the edges to remove edge artifacts and save to temp folder
        raster = rxr.open_rasterio(file).squeeze()
        clipped = raster.isel(x=slice(buffer, -buffer), y=slice(buffer, -buffer))
        
        # if the raster name starts with 17, it's in UTM 17, otherwise it's in UTM 18
        utm_zone = int(file.name[:2])
        if utm_zone == 17:
            clipped = clipped.rio.clip([geom_utm17], all_touched=True)
        elif utm_zone == 18:
            clipped = clipped.rio.clip([geom_utm18], all_touched=True)
        else:
            raise ValueError(f'Unexpected UTM zone in filename: {file.name}')
        
        clipped.rio.to_raster(temp_out_path / file.name, compress='LZW', tiled=True)
        with rio.open(temp_out_path / file.name, 'r+') as src:
            if cmap is not None:
                src.write_colormap(1, cmap)
            src.build_overviews(get_overview_levels(src), Resampling.nearest)
            
    files = list(Path('D:/s2_out/').glob(f'{folder}/*.tif'))
    Parallel(n_jobs=2)(delayed(process_file)(file) for file in tqdm(files, desc=f'Processing {folder}'))

ValueError: NULL color table

In [19]:
file.name[:2]

'18'

In [ ]:

try:
    client = Client()
except NameError:
    pass
client = Client()
display(client)


footprint_gdf = gpd.read_file(r'd:\chesapeake_bay_2022_2024edition\footprint.gpkg')
target_crs = footprint_gdf.crs

colormap = rio.open('../runs/s2_out/2020/17SMB.tif').colormap(1)

# for all the years and temp files in the temp folders, reproject them to the target CRS, and mask to the footprint

def reproject_and_mask(file):
        raster = rxr.open_rasterio(file, chunks=(1, 4096, 4096)).squeeze()
        raster_reprojected = raster.rio.reproject(target_crs)
        # mask to footprint
        raster_masked = raster_reprojected.rio.clip(footprint_gdf.geometry, all_touched=True)
        out_path = temp_out_path / file.name
        raster_masked.rio.to_raster(out_path, compress='LZW', tiled=True)
        # add colormap if applicable
        if folder != 'transitions':
            with rio.open(out_path, 'r+') as dst:
                dst.write_colormap(1, colormap)
                # dst.build_overviews([2, 4, 8, 16], Resampling.nearest)
                
for folder in ['2020', '2021', '2022', '2023', '2024']:
    temp_out_path = Path('../runs/s2_out/') / folder / 'temp_reprojected'
    temp_out_path.mkdir(parents=True, exist_ok=True)
    
    temp_files = list((Path('../runs/s2_out/') / folder / 'temp').glob('*.tif'))
    
    # Parallel(n_jobs=2, backend='threading')(delayed(reproject_and_mask)(file) for file in tqdm(temp_files))    
    for file in tqdm(temp_files):
        reproject_and_mask(file)


c:\Users\dh2306\projects\s2flow\.venv\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 61779 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:61779/status,
Dashboard: http://127.0.0.1:61779/status,Workers: 6
Total threads: 24,Total memory: 127.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61782,Workers: 0
Dashboard: http://127.0.0.1:61779/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61814,Total threads: 4
Dashboard: http://127.0.0.1:61815/status,Memory: 21.30 GiB
Nanny: tcp://127.0.0.1:61785,


  2%|▏         | 1/50 [04:15<3:28:58, 255.90s/it]2026-02-13 17:46:02,883 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:61748', name: 0, status: running, memory: 0, processing: 0>
2026-02-13 17:46:02,905 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:61750', name: 2, status: running, memory: 0, processing: 0>
2026-02-13 17:46:02,957 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:61756', name: 1, status: running, memory: 0, processing: 0>
2026-02-13 17:46:02,964 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <WorkerState 'tcp://127.0.0.1:61763', name: 5, status: running, memory: 0, processing: 0>
2026-02-13 17:46:02,966 - distributed.scheduler - WARNING - Worker failed to heartbeat for 309s; attempting restart: <Worke

In [17]:
# for each year - get the distribution of land cover from the masked rasters in temp
import json

def get_counts(file):
    with rio.open(file) as src:
        data = src.read(1)
        counts = np.zeros(5, dtype=np.uint64)
        for i in range(1, 6):
            counts[(i-1)] = np.sum(data == i)
    return counts

# use multithraeding to get counts for all the files in temp for each year, and sum them up to get the total distribution of land cover for each year
# for folder in ['2020', '2021', '2022', '2023', '2024']:
for folder in ['2025']:
    temp_out_path = Path('../runs/s2_out/') / folder / 'temp'
    temp_files = list(temp_out_path.glob('*.tif'))
    counts_list = Parallel(n_jobs=4, backend='threading')(delayed(get_counts)(file) for file in tqdm(temp_files))
    lc_counts = np.sum(counts_list, axis=0)
    lc_counts_dict = {i: int(lc_counts[i-1]) for i in range(1, 6)}
    print(f'Year: {folder}, Land Cover Distribution: {lc_counts / np.sum(lc_counts)}')
    with open(Path('../runs/s2_out') / folder / 'lc_distribution.json', 'w') as f:
        json.dump(lc_counts_dict, f)















100%|██████████| 50/50 [02:18<00:00,  2.77s/it]


Year: 2025, Land Cover Distribution: [0.09583261 0.26533696 0.59800727 0.03866563 0.00215753]


In [ ]:
# build overview

In [18]:
# put these in a CSV
import pandas as pd

dicts = []
for folder in ['2020', '2021', '2022', '2023', '2024', '2025']:
    with open(Path('../runs/s2_out') / folder / 'lc_distribution.json', 'r') as f:
        lc_counts_dict = json.load(f)
    print(f'Year: {folder}, Land Cover Distribution: {lc_counts_dict}')
    lc_counts_dict['year'] = folder
    dicts.append(lc_counts_dict)

dists_df = pd.DataFrame(dicts)
# rename columns to class names
dists_df = dists_df.rename(columns={'1': 'Open Water', '2': 'Herbaceous', '3': 'Forest Canopy', '4': 'Impervious', '5': 'Barren Land'})
dists_df.to_csv('../runs/s2_out/lc_distributions.csv', index=False)

dists_df

Year: 2020, Land Cover Distribution: {'1': 9274086378, '2': 26183948482, '3': 56884161919, '4': 3756911335, '5': 231541806}
Year: 2021, Land Cover Distribution: {'1': 9243231040, '2': 26233670638, '3': 56813904625, '4': 3789428123, '5': 250415494}
Year: 2022, Land Cover Distribution: {'1': 9264781103, '2': 26706258866, '3': 56477151879, '4': 3685164919, '5': 198171073}
Year: 2023, Land Cover Distribution: {'1': 9308284166, '2': 25991032030, '3': 57088068046, '4': 3756241004, '5': 187551426}
Year: 2024, Land Cover Distribution: {'1': 9262738091, '2': 25565140799, '3': 57550785848, '4': 3739923564, '5': 212412786}
Year: 2025, Land Cover Distribution: {'1': 9231701563, '2': 25560314500, '3': 57606954316, '4': 3724719433, '5': 207838028}


,Open Water,Herbaceous,Forest Canopy,Impervious,Barren Land,year
0,9274086378,26183948482,56884161919,3756911335,231541806,2020
1,9243231040,26233670638,56813904625,3789428123,250415494,2021
2,9264781103,26706258866,56477151879,3685164919,198171073,2022
3,9308284166,25991032030,57088068046,3756241004,187551426,2023
4,9262738091,25565140799,57550785848,3739923564,212412786,2024
5,9231701563,25560314500,57606954316,3724719433,207838028,2025


In [5]:
def build_pyramids(path):
    with rio.open(path, 'r+') as src:
        src.build_overviews([2, 4, 8, 16], Resampling.nearest)
    
for year in []:
    temp_out_path = Path('../runs/s2_out/') / year / 'temp'
    temp_files = list(temp_out_path.glob('*.tif'))
    Parallel(n_jobs=4, backend='threading')(delayed(build_pyramids)(file) for file in tqdm(temp_files))

In [7]:
from rio_vrt import build_vrt

for year in tqdm(['2020', '2021', '2022', '2023', '2024', '2025']):
    # create 2 vrts - one for UTM 17 and one for UTM 18 (use first 2 digits of filename to determine)
    temp_out_path = Path('D:/s2_out/') / year / 'temp'
    temp_files = list(temp_out_path.glob('*.tif'))
    utm17_files = [file for file in temp_files if file.name.startswith('17')]
    utm18_files = [file for file in temp_files if file.name.startswith('18')]
    build_vrt(temp_out_path / f'{year}_utm17.vrt', utm17_files)
    build_vrt(temp_out_path / f'{year}_utm18.vrt', utm18_files)

100%|██████████| 6/6 [00:42<00:00,  7.07s/it]


In [8]:
# now build pyramids for the VRTs
for year in tqdm(['2020', '2021', '2022', '2023', '2024', '2025']):
    temp_out_path = Path('D:/s2_out/') / year / 'temp'
    for vrt_file in temp_out_path.glob('*.vrt'):
        build_pyramids(vrt_file)

 33%|███▎      | 2/6 [28:49<57:34, 863.64s/it]  

: 

: 